### **Parameterization Cell**

In [0]:
dbutils.widgets.text("File name","")

In [0]:
file_name =dbutils.widgets.get("File name") 

### **Data Reading**

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@databrcks.dfs.core.windows.net/orders")
display(df.limit(5))


order_id,customer_id,product_id,order_date,quantity,total_amount,_rescued_data
O00001,C00710,P0159,2023-03-22,3,2022.87,null
O00002,C00954,P0036,2023-06-30,2,3560.74,null
O00003,C01578,P0427,2023-11-06,3,5903.52,null
O00004,C00962,P0332,2024-02-27,3,4107.99,null
O00005,C00156,P0038,2024-10-13,5,5784.95,null


In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation",f'abfss://bronze@databrcks.dfs.core.windows.net/checkpoint_{file_name}')\
    .load(f"abfss://source@databrcks.dfs.core.windows.net/{file_name}")


### **Data Writing**

In [0]:
df.writeStream.format("parquet")\
    .outputMode("append")\
    .option("checkpointLocation", f"abfss://bronze@databrcks.dfs.core.windows.net/checkpoint_{file_name}")\
    .option("path",f"abfss://bronze@databrcks.dfs.core.windows.net/{file_name}")\
    .trigger(once=True)\
    .start()